# Analyse de la marchabilité à Genève à partir des données du Panel Lémanique

Ce notebook a pour objectif de fournir des analyses statistiques et cartographique sur la marche à Genève à partir des données GPS du Panel Lémanique

Les données utilisées proviennent directement des dumps pour le suivi GPS du PL, et non des données de Matrices OD

## Imports 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import folium
from folium import Choropleth, GeoJson
from folium.plugins import Fullscreen
import gzip
from functools import reduce
import geopandas as gpd
from shapely.geometry import Point
from branca.colormap import linear
import seaborn as sns
import matplotlib.dates as mdates
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from folium.plugins import HeatMap
from branca.element import Element
import datetime
import branca.colormap as cm
from shapely.geometry import LineString
import h3

In [ ]:
path_data = '../Data/'
path_data_pl = '../Data/20241120_dump_situee\\'
path_save_data = '../Data/generated'

In [ ]:
users_pl = pd.read_csv(path_data_pl + '241120_user_statistics.csv', index_col = 0)
legs_pl =  pd.read_pickle(path_data + 'legs.pkl')
#legs_3weeks =  pd.read_csv(path_data_pl + 'legs_3weeks.csv') #Si on veut avoir uniquement les personnes ayant 3 semaine complètes de suivi (l'échantillon est très petit)

In [ ]:
h3_resolution = 9

In [ ]:
users_pl_ge = users_pl[users_pl['KT_home_survey'] == 'GE']
user_id_pl_ge = users_pl_ge['user_id_fors']

## Traitement des données

Traitement des legs

In [ ]:
#exclusion des outliers
legs_pl = legs_pl[legs_pl['extreme99_length_mode'] == False]  

#conversion en km 
legs_pl['length_leg'] /= 1000 

Statistiques sur les legs à Genève

In [ ]:
#filtre aux résident de Genève
legs_pl_ge = legs_pl[legs_pl['KT_home_survey'] == 'GE']

#regroupement par utilisateur
legs_pl_ge_user = legs_pl_ge.groupby('user_id_fors').agg({'length_leg': 'sum'}).reset_index()

Statistiques sur les legs en marche à Genève

In [ ]:
#filtre des legs effectuées à pied
legs_pl_marche = legs_pl[legs_pl['mode'] == 'Mode::Walk'].copy()
legs_pl_marche.rename(columns = {'length_leg': 'length_marche'}, inplace = True)

#filtre à Genève
legs_pl_marche_ge = legs_pl_marche[legs_pl_marche['KT_home_survey'] == 'GE']

#regroupement par utilisateur
legs_pl_marche_ge_user =  legs_pl_marche_ge.groupby('user_id_fors').agg({'length_marche': 'sum'}).reset_index()

Création d'un seul DF avec la distance totale marchée, la distance totale déplacée et le nombre de jours de suivi

In [ ]:
#ajout du nombre de jours de suivi
legs_pl_marche_ge_user = legs_pl_marche_ge_user.merge(users_pl_ge[['user_id_fors', 'days_in_range', 'age_fr', 'gdr']], on = 'user_id_fors')

#ajout de la distance totale déplacée
legs_pl_marche_ge_user = legs_pl_marche_ge_user.merge(legs_pl_ge_user[['user_id_fors', 'length_leg']], on = 'user_id_fors')

#Calcul de la moyenne quotidienne
legs_pl_marche_ge_user['dist_marche_quoti'] = legs_pl_marche_ge_user['length_marche'] / legs_pl_marche_ge_user['days_in_range']

Création d'un df avec la part modale de la marche pondérée par le nombre de jours de suivi pour chaque personne

In [ ]:
pm_marche_user = legs_pl_marche_ge_user.copy()
pm_marche_user['pm_marche'] = pm_marche_user['length_marche'] / pm_marche_user['length_leg']
pm_marche_user['pm_marche_weighted'] = pm_marche_user['pm_marche'] * pm_marche_user['days_in_range']

In [ ]:
#Groupement du df par catégorie d'âge
pm_marche_user_age= pm_marche_user.groupby(['age_fr']).agg({'pm_marche_weighted': 'sum', 'days_in_range' : 'sum'}).reset_index()
pm_marche_user_age['pm_marche'] = pm_marche_user_age['pm_marche_weighted'] / pm_marche_user_age['days_in_range'] *100

In [ ]:
#Groupement du df par genre et catégorie d'âge
pm_marche_user_homme = pm_marche_user[pm_marche_user['gdr'] == 'Homme']
pm_marche_user_femme = pm_marche_user[pm_marche_user['gdr'] == 'Femme']

pm_marche_user_homme_age= pm_marche_user_homme.groupby(['age_fr']).agg({'pm_marche_weighted': 'sum', 'days_in_range' : 'sum'}).reset_index()
pm_marche_user_homme_age['pm_marche'] = pm_marche_user_homme_age['pm_marche_weighted'] / pm_marche_user_homme_age['days_in_range'] *100

pm_marche_user_femme_age= pm_marche_user_femme.groupby(['age_fr']).agg({'pm_marche_weighted': 'sum', 'days_in_range' : 'sum'}).reset_index()
pm_marche_user_femme_age['pm_marche'] = pm_marche_user_femme_age['pm_marche_weighted'] / pm_marche_user_femme_age['days_in_range'] *100

In [ ]:
#Groupement du df par genre
pm_marche_user_gdr= pm_marche_user.groupby(['gdr']).agg({'pm_marche_weighted': 'sum', 'days_in_range' : 'sum'}).reset_index()
pm_marche_user_gdr['pm_marche'] = pm_marche_user_gdr['pm_marche_weighted'] / pm_marche_user_gdr['days_in_range'] *100

In [ ]:
legs_pl_marche_ge_user = legs_pl_marche_ge_user.sort_values(by='age_fr')

## Visualisation des données

### Boxplot de la distance marchée par genre et classe d'âge

In [ ]:
plt.figure()
fig, ax1 = plt.subplots()
sns.boxplot(legs_pl_marche_ge_user, y = 'dist_marche_quoti', x = 'age_fr', hue = 'gdr')

plt.title('Répartition de la distance moyenne marchée par les résidents du canton de Genève, par classe d\'âge et par genre')
plt.ylabel('Distance moyenne quotidienne marchée')
plt.ylim(0, 10)
plt.grid()

### Boxplot de la distance marchée par classe d'âge

In [ ]:
plt.figure()

fig, ax1 = plt.subplots()
sns.boxplot(legs_pl_marche_ge_user, y = 'dist_marche_quoti', x = 'age_fr', ax = ax1)

ax2 = ax1.twinx()
ax2.plot(pm_marche_user_age['age_fr'], pm_marche_user_age['pm_marche'], 'ro') #, label='Pourcentage des distances totales'
ax2.set_ylabel('Part modale de la marche (% des distances)')

plt.title('Répartition de la distance moyenne marchée par les résidents du canton de Genève, par classe d\'âge')
ax1.set_ylim(top=10)
ax1.set_ylabel('Distance moyenne quotidienne marchée')
plt.grid()
ax1.set_ylim(bottom=0)
ax2.set_ylim(bottom=0)


plt.legend()

### Boxplot de la distance marchée par genre

In [ ]:
plt.figure()
fig, ax1 = plt.subplots()
sns.boxplot(legs_pl_marche_ge_user, y = 'dist_marche_quoti', x = 'gdr', ax = ax1)

ax1.set_ylim(top=10)
ax1.set_ylabel("Distance moyenne quotidienne marchée")
ax1.set_ylim(bottom=0)

ax2 = ax1.twinx()
ax2.plot(pm_marche_user_gdr['gdr'], pm_marche_user_gdr['pm_marche'], 'ro', label='Pourcentage des distances totales')
ax2.set_ylabel('Part modale de la marche (% des distances)') 
ax2.set_ylim(bottom=0)

plt.title('Répartition de la distance moyenne marchée par les résidents du canton de Genève, par genre')
plt.grid()
plt.legend()

Boxplot de la distance marchée par genre et classe d'âge

In [ ]:
fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(14, 6))  # 1 ligne, 2 colonnes

# Graphique pour les hommes
sns.boxplot(
    legs_pl_marche_ge_user[legs_pl_marche_ge_user['gdr'] == 'Homme'],
    y='dist_marche_quoti', x='age_fr', ax=ax1
)
ax2 = ax1.twinx()
ax2.plot(
    pm_marche_user_homme_age['age_fr'],
    pm_marche_user_homme_age['pm_marche'],
    'ro', label='Pourcentage des distances totales'
)
#ax2.set_ylabel('Part modale de la marche (% des distance)')
ax1.set_ylabel('Distance moyenne quotidienne marchée')
ax1.set_ylim(0, 10)
ax2.set_ylim(0, 20)
ax1.set_title('Hommes')
ax1.grid()
ax2.legend(loc='upper left')
ax2.set_ylabel('Part modale de la marche (% des distances)')

# Graphique pour les femmes
sns.boxplot(
    legs_pl_marche_ge_user[legs_pl_marche_ge_user['gdr'] == 'Femme'],
    y='dist_marche_quoti', x='age_fr', ax=ax3
)
ax4 = ax3.twinx()
ax4.plot(
    pm_marche_user_femme_age['age_fr'],
    pm_marche_user_femme_age['pm_marche'],
    'ro', label='Pourcentage des distances totales'
)

ax3.set_ylabel('Distance moyenne quotidienne marchée')
ax3.set_ylim(0, 10)
ax4.set_ylim(0, 20)
ax3.set_title('Femmes')
ax3.grid()
ax4.legend(loc='upper left')
ax4.set_ylabel('Part modale de la marche (% des distances)')

fig.suptitle("Répartition de la distance moyenne quotidienne marchée par les résidents du canton de Genève")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Heatmap de la fréquentation piétonnne (legs)

In [ ]:
#filtre des legs: après avoir filtré les legs réalisées par les résidents du canton, on filtre mtn aux legs réalisées dans le Canton
legs_pl_marche_ge_ge = legs_pl_marche_ge[legs_pl_marche_ge['intra_GE'] == 1]

In [ ]:
#Tentative de rajouter des points entre les points des linestrings. 

#n = 2  # nombre de points supplémentaires
#all_points = []
#
#for line in legs_pl_marche_ge_ge['geometry']:
#    if len(line.coords) < 2:
#        continue  # Ignore les LineString trop courtes
#    num_segments = len(line.coords) - 1
#    for i in range(num_segments):
#        start = line.coords[i]
#        end = line.coords[i + 1]
#        if i == 0:
#            all_points.append([start[1], start[0]])
#        for j in range(1, n + 1):
#            frac = j / (n + 1)
#            interp_point = LineString([start, end]).interpolate(frac, normalized=True)
#            all_points.append([interp_point.y, interp_point.x])
#        all_points.append([end[1], end[0]])

In [ ]:
#Création d'une liste des points fréquentés: tous les points des linestrings des legs
all_points_frequentation = []
for line in legs_pl_marche_ge_ge['geometry']:
    for point in line.coords:
        all_points_frequentation.append([point[1], point[0]]) 

In [ ]:
#Conversion des points en cellules H3
h3_indices_frequentation = [h3.latlng_to_cell(lat, lon, h3_resolution) for lat, lon in all_points_frequentation]
h3_counts_frequentation = pd.Series(h3_indices_frequentation).value_counts().reset_index()
h3_counts_frequentation.columns = ['index', 'count']

h3_counts_frequentation['lat'] = h3_counts_frequentation['index'].apply(lambda h: h3.cell_to_latlng(h)[0])
h3_counts_frequentation['lon'] = h3_counts_frequentation['index'].apply(lambda h: h3.cell_to_latlng(h)[1])

heat_data_frequentation = h3_counts_frequentation[['lat', 'lon', 'count']].values.tolist()

In [ ]:
#Création de la heatmap
m = folium.Map(location=[46.2044, 6.1432], zoom_start=11, tiles="cartodbpositron")

colormap = cm.linear.YlGn_09.scale(h3_counts_frequentation['count'].min(), h3_counts_frequentation['count'].max())

for _, row in h3_counts_frequentation.iterrows():
    hex_boundary = h3.cell_to_boundary(row['index'])
    folium.Polygon(
        locations=hex_boundary,
        color=None,
        fill=True,
        fill_color=colormap(row['count']),
        fill_opacity=0.7,
        weight=0
    ).add_to(m)

# 4. Ajouter la légende
colormap.caption = 'Fréquentation'
colormap.add_to(m)

# 4. Ajouter la heatmap
HeatMap(heat_data_frequentation, radius=15, max_zoom=13).add_to(m)

In [ ]:
#Affichage de la heatmap 
#m

### Heatmap de la fréquentation piétonne (destination)

In [ ]:
#Création d'une liste avec toutes les destinations des trajets à pied
all_dest_liste = [
    [line.coords[-1][1], line.coords[-1][0]]  # [lat, lon]
    for line in legs_pl_marche_ge_ge['geometry']
]

In [ ]:
#Conversion de la liste en cellules h3
h3_indices_destinations = [h3.latlng_to_cell(lat, lon, h3_resolution) for lat, lon in all_dest_liste]
h3_counts_destination = pd.Series(h3_indices_destinations).value_counts().reset_index()
h3_counts_destination.columns = ['index', 'count']

h3_counts_destination['lat'] = h3_counts_destination['index'].apply(lambda h: h3.cell_to_latlng(h)[0])
h3_counts_destination['lon'] = h3_counts_destination['index'].apply(lambda h: h3.cell_to_latlng(h)[1])

heat_data_destination = h3_counts_destination[['lat', 'lon', 'count']].values.tolist()


In [ ]:
#Création de la carte
m = folium.Map(location=[46.2044, 6.1432], zoom_start=11, tiles="cartodbpositron")

colormap = cm.linear.YlGn_09.scale(h3_counts_destination['count'].min(), h3_counts_destination['count'].max())

for _, row in h3_counts_destination.iterrows():
    hex_boundary = h3.cell_to_boundary(row['index'])
    folium.Polygon(
        locations=hex_boundary,
        color=None,
        fill=True,
        fill_color=colormap(row['count']),
        fill_opacity=0.7,
        weight=0
    ).add_to(m)

# 4. Ajouter la légende
colormap.caption = 'Fréquentation'
colormap.add_to(m)

# 4. Ajouter la heatmap
HeatMap(heat_data_destination, radius=15, max_zoom=13).add_to(m)

In [ ]:
#Affichage de la map
#m